# Noise Resilience of Dimensionality Reduction for HAR
CSE 555 Fall 2026, Track 2. This notebook is only a driver: all logic is in `har_noise/` and `run_experiments.py`.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

import run_experiments

# Run the full pipeline only if results are missing (about 5 minutes).
if not Path("results/summary.csv").exists():
    run_experiments.main()

: 

In [ ]:
info = json.loads(Path("results/run_info.json").read_text())
print("versions:", info["python"], "numpy", info["numpy"], "sklearn", info["sklearn"])
print("rank of within-class scatter Sw:", info["rank_Sw"], "of 561")
print("leakage check:", info["leakage"])
print("PCA cumulative variance:", info["pca_cum_var"])
print("chosen hyperparameters:")
for p, b in info["best_params"].items():
    print(f"  {p:12s} {b}")

In [ ]:
summary = pd.read_csv("results/summary.csv")
main = ["none", "pca50", "ica50", "lda5", "pca100-lda5"]
cols = ["projection", "classifier", "clean_acc", "acc_30", "acc_50", "slope_per10", "breakdown", "fisher_ratio_50"]
for corr in ["gaussian", "dropout", "ood"]:
    print(corr)
    display(summary[(summary.corruption == corr) & summary.projection.isin(main)][cols].round(3))

In [ ]:
shrink = pd.read_csv("results/shrinkage_sweep.csv")
display(shrink.groupby(["shrinkage", "level"])["acc"].mean().unstack().round(3))

In [ ]:
# Robustness ranking: mean accuracy over every corrupted condition
# (3 corruption types x 5 non-zero levels, averaged over seeds), next to clean accuracy.
res = pd.read_csv("results/results.csv")
noisy = res[res.level > 0].groupby(["corruption", "level", "projection", "classifier"])["acc"].mean()
rank = noisy.groupby(["projection", "classifier"]).mean().rename("robust_acc").to_frame()
rank["gaussian_50"] = noisy.loc["gaussian", 0.5]
rank["dropout_avg"] = noisy.loc["dropout"].groupby(["projection", "classifier"]).mean()
rank["clean_acc"] = res[res.level == 0].groupby(["projection", "classifier"])["acc"].mean()
rank = rank.sort_values("robust_acc", ascending=False).reset_index()
rank.index = rank.index + 1
rank.to_csv("results/ranking.csv")
display(rank.round(3))

In [ ]:
for f in sorted(Path("figures").glob("*.png")):
    print(f.name)
    display(Image(filename=str(f)))